In [ ]:
from pathlib import Path
from pyspark.sql.session import SparkSession
from pyspark.sql.types import IntegerType

def find_project_root():
    start_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
    for candidate in (start_dir, *start_dir.parents):
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find the project data directory from {start_dir}")

project_root = find_project_root()
warehouse_path = (project_root / "data" / "warehouse").resolve()

def get_spark_session(appName="default"):
    spark = (
        SparkSession
        .builder
        .appName(appName)
        .master("local[*]")
        .config("spark.ui.port", "4142")
        .config("spark.port.maxRetries", "20") 
        .config("spark.sql.warehouse.dir", warehouse_path.as_uri())
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    spark.sql("CREATE DATABASE IF NOT EXISTS sujeet_db")
    spark.sql("USE sujeet_db")
    return spark


def stop_start(app_name):
    global spark

    # 1. Safely teardown the old session if present
    if 'spark' in globals() and spark is not None:
        try:
            spark.stop()
            print("Old Spark session successfully stopped.")
        except Exception as e:
            print(f"Failed to stop active session cleanly. Error: {e}")
        finally:
            spark = None  # Completely unbind the dead object from global memory
    else:
        print("No active 'spark' session found to stop.")

    # 2. Return the fresh session
    print(f"Starting new Spark session: '{app_name}'...")
    return get_spark_session(app_name)

# Initialize your fresh execution context
spark = stop_start("working_with_different_data_type")


No active 'spark' session found to stop.
Starting new Spark session: 'working_with_different_data_type'...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/11 14:08:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/11 14:08:39 WARN Utils: Service 'SparkUI' could not bind on port 4142. Attempting port 4143.


In [ ]:
import os
from pyspark.sql import SparkSession

class LocalDataFrameLoader:
    database_name = "sujeet_db"
    def __init__(self) -> None:
        base_dir = str(project_root)
        
        self.flight_path = os.path.abspath(os.path.join(base_dir, "data/flight-data/csv")).replace("\\", "/")
        self.retail_path = os.path.abspath(os.path.join(base_dir, "data/retail-data")).replace("\\", "/")

    def create_flight_and_retail_data(self, spark: SparkSession) -> None:
        spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.database_name}")
        spark.sql(f"USE {self.database_name}")

        # Verify paths exist before Spark tries to read them
        for name, path in [("flight", self.flight_path), ("retail", self.retail_path)]:
            if not os.path.exists(path):
                raise FileNotFoundError(f"{name} directory not found at: {path} (Current working dir: {os.getcwd()})")
            print(f"{name} path verified ({len(os.listdir(path))} items): {path}")

        print("Loading flight data...")
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.database_name}.flight
            USING csv
            OPTIONS (
                path '{self.flight_path}',
                header 'true',
                inferSchema 'true'
            )
        """)

        print("Loading retail data...")
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.database_name}.retail
            USING csv
            OPTIONS (
                path '{self.retail_path}',
                header 'true',
                inferSchema 'true',
                recursiveFileLookup 'true'
            )
        """)


In [3]:
from pyspark.sql import DataFrame, SparkSession


class CreateSampleData(LocalDataFrameLoader):

    def _replace_table(
        self,
        spark: SparkSession,
        table_name: str,
        schema: str,
        values: str,
    ) -> None:
        qualified_table_name = f"{self.database_name}.{table_name}"
        warehouse_path = spark.conf.get("spark.sql.warehouse.dir").rstrip("/")
        table_path = f"{warehouse_path}/{self.database_name}.db/{table_name}"

        spark.sql(f"CREATE DATABASE IF NOT EXISTS {self.database_name}")
        spark.sql(f"USE {self.database_name}")
        spark.sql(f"DROP TABLE IF EXISTS {qualified_table_name}")
        spark.sql(
            f"CREATE TABLE {qualified_table_name} ({schema}) "
            f"USING PARQUET LOCATION '{table_path}'"
        )
        spark.sql(f"INSERT OVERWRITE TABLE {qualified_table_name} VALUES {values}")

    def prepare_data(self, spark: SparkSession) -> DataFrame:
        self._replace_table(
            spark,
            table_name="department",
            schema="name STRING, department STRING, salary INT",
            values="""
                ('James', 'Sales', 3000),
                ('Michael', 'Sales', 4600),
                ('Robert', 'Sales', 4100),
                ('Maria', 'Finance', 3000),
                ('James', 'Sales', 3000),
                ('Scott', 'Finance', 3300),
                ('Jen', 'Finance', 3900),
                ('Jeff', 'Marketing', 3000),
                ('Kumar', 'Marketing', 2000),
                ('Saif', 'Sales', 4100),
                ('Daniel', NULL, NULL),
                ('Christian', NULL, NULL)
            """,
        )
        department_df = spark.table(f"{self.database_name}.department")
        return department_df

    def create_payment_table(self, spark: SparkSession) -> None:
        self._replace_table(
            spark,
            table_name="payment",
            schema="payment_amount DECIMAL(8, 2), payment_date STRING, store_id INT",
            values="""
                (1200.99, '2018-01-18', 1),
                (189.23, '2018-02-15', 1),
                (33.43, '2018-03-03', 3),
                (7382.10, '2019-01-11', 2),
                (382.92, '2019-02-18', 1),
                (322.34, '2019-03-29', 2),
                (2929.14, '2020-01-03', 2),
                (499.02, '2020-02-19', 3),
                (994.11, '2020-03-14', 1),
                (394.93, '2021-01-22', 2),
                (3332.23, '2021-02-23', 3),
                (9499.49, '2021-03-10', 3),
                (3002.43, '2018-02-25', 2),
                (100.99, '2019-03-07', 1),
                (211.65, '2020-02-02', 1),
                (500.73, '2021-01-06', 3)
            """,
        )

    def create_sales_table(self, spark: SparkSession) -> None:
        self._replace_table(
            spark,
            table_name="sales_metrics",
            schema="day_id INT, daily_sales DECIMAL(8, 2)",
            values="""
                (1, 2.00),
                (2, 4.00),
                (3, 4.00),
                (4, 4.00),
                (5, 6.00)
            """,
        )

    def create_skewed_table(self, spark: SparkSession) -> None:
        self._replace_table(
            spark,
            table_name="skew_metrics",
            schema="id INT, item_value DECIMAL(8, 2)",
            values="""
                (1, 2.00),
                (2, 2.00),
                (3, 2.00),
                (4, 2.00),
                (5, 12.00)
            """,
        )

    def create_correlation_table(self, spark: SparkSession) -> None:
        self._replace_table(
            spark,
            table_name="weather_sales",
            schema=(
                "day_id INT, temperature DECIMAL(8, 2), "
                "ice_cream_sales DECIMAL(8, 2)"
            ),
            values="""
                (1, 2.00, 10.00),
                (2, 4.00, 20.00),
                (3, 4.00, 25.00),
                (4, 4.00, 25.00),
                (5, 6.00, 40.00)
            """,
        )

In [4]:
sample_data = CreateSampleData()
sample_data.prepare_data(spark)
sample_data.create_payment_table(spark)
sample_data.create_sales_table(spark)
sample_data.create_skewed_table(spark)
sample_data.create_correlation_table(spark)
sample_data.create_flight_and_retail_data(spark)

flight path verified (6 items): /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/flight-data/csv
retail path verified (7 items): /home/suanki/Desktop/projects/suanki-pyspark-and-optimization/data/retail-data
Loading flight data...
Loading retail data...


In [ ]:
spark.sql("SHOW TABLES IN sujeet_db").show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|sujeet_db|   department|      false|
|sujeet_db|       flight|      false|
|sujeet_db|      payment|      false|
|sujeet_db|       retail|      false|
|sujeet_db|sales_metrics|      false|
|sujeet_db| skew_metrics|      false|
|sujeet_db|weather_sales|      false|
+---------+-------------+-----------+



In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, ArrayType
)


print("@@@@@@@@@@@@@@@@@")

print("creating dataset 'emp_df' and 'dept_df' for join playbook")

print("@@@@@@@@@@@@@@@@@@")
# 1. Employee Schema (Includes StructType for 'details' and ArrayType for 'skills')
emp_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("dept_id", IntegerType(), True),
    StructField("details", StructType([
        StructField("location", StringType(), True),
        StructField("level", StringType(), True)
    ]), True),
    StructField("skills", ArrayType(StringType()), True)
])

emp_data = [
    (1, "Amit", 101, ("Pune", "L3"), ["Spark", "Scala", "AWS"]),
    (2, "Priya", 102, ("Bengaluru", "L2"), ["Python", "SQL"]),
    (3, "Rohan", 101, ("Mumbai", "L1"), ["Java", "Spark"]),
    (4, "Sneha", 105, ("Pune", "L2"), ["PySpark", "Airflow"]),  # 105 does NOT exist in dept_df
    (5, "Vikram", None, ("Delhi", "L4"), ["Kafka", "Scala"])    # NULL dept_id
]

# 2. Department Schema (Includes StructType for 'hq_info' and ArrayType for 'tech_stack')
dept_schema = StructType([
    StructField("dept_id", IntegerType(), True),
    StructField("dept_name", StringType(), True),
    StructField("hq_info", StructType([
        StructField("city", StringType(), True),
        StructField("building", StringType(), True)
    ]), True),
    StructField("req_skills", ArrayType(StringType()), True)
])

dept_data = [
    (101, "Data Engineering", ("Pune", "Tower A"), ["Spark", "Scala", "Kafka"]),
    (102, "Analytics", ("Bengaluru", "Tower B"), ["Python", "SQL", "Tableau"]),
    (103, "Cloud Ops", ("Hyderabad", "Tower C"), ["AWS", "Terraform", "Docker"]), # No employees
    (104, "Security", ("Mumbai", "Tower A"), ["Linux", "Networking"])             # No employees
]

emp_df = spark.createDataFrame(emp_data, schema=emp_schema)
dept_df = spark.createDataFrame(dept_data, schema=dept_schema)

emp_df.printSchema()
dept_df.printSchema()